# drugs.ipynb — Clean & Normalize Epiconsult Drugs Price List

This notebook:
- Reads `drugs.csv` **or** `drugs.xlsx` from your **project root**
- Detects each **category** from rows like **"Listing of ..."** (removes the word "Listing of")
- Builds one clean table with columns ordered as: **S/N, Name, Category, (price columns...)**
- Converts any **blank / zero / < 100** values in price columns to **`N/A`**
- Formats valid prices as **₦#,###.##** (true Naira sign)
- Exports to **`drugs_new.csv`** (UTF‑8 with BOM so ₦ displays correctly on Windows/Excel)


In [1]:
# --- Cell 1: Setup ---
from pathlib import Path
import csv
import re
import pandas as pd
import numpy as np
import tempfile
import os

# Input (in your project root)
CSV_IN  = Path("drugs.csv")
XLSX_IN = Path("drugs.xlsx")

# Output (keep just one cleaned file)
OUTPUT_CSV = Path("drugs_new.csv")

# Rules
BLANK_TO_NA = True
ZERO_TO_NA = True
BELOW_100_TO_NA = True   # ✅ your requirement
FORMAT_NAIRA = True


In [2]:
# --- Cell 2: Read raw rows (works for CSV or Excel) ---

def read_rows_from_csv(path: Path):
    raw_text = path.read_text(encoding="utf-8", errors="replace").splitlines()
    rows = []
    reader = csv.reader(raw_text)
    for r in reader:
        rows.append([(c or "").strip() for c in r])
    return rows

def read_rows_from_excel(path: Path):
    # Read everything as text-ish, preserve blanks
    tmp = pd.read_excel(path, header=None, dtype=object)
    tmp = tmp.replace({np.nan: ""})
    rows = []
    for _, r in tmp.iterrows():
        rows.append([str(c).strip() if c != "" else "" for c in r.tolist()])
    return rows

if CSV_IN.exists():
    INPUT_PATH = CSV_IN
    rows = read_rows_from_csv(CSV_IN)
elif XLSX_IN.exists():
    INPUT_PATH = XLSX_IN
    rows = read_rows_from_excel(XLSX_IN)
else:
    raise FileNotFoundError(
        "Neither drugs.csv nor drugs.xlsx was found in your project root. "
        f"Expected one of: {CSV_IN.resolve()} or {XLSX_IN.resolve()}"
    )

print("Using:", INPUT_PATH.resolve())
print("Total raw rows:", len(rows))
print("Sample rows (first 10):")
for i in range(min(10, len(rows))):
    print(rows[i])


Using: C:\Users\ACEMX\Desktop\LLMprojects\e-CLINIC\drugs.xlsx
Total raw rows: 2145
Sample rows (first 10):
['Epiconsult Diagnostics - Service Charges Listing as at 31-Oct-2025', '', '', '', '']
['Drugs Service Charges Listing as at 31-Oct-2025', '', '', '', '']
['', '', '', '', '']
['Listing of ANAESTHETICS/ANXIOLYTICS/EMERGENCY DRUGS', '', '', '', '']
['', '', '', '', '']
['S/N', 'Name', 'Outsourced (B)', 'Walk in Patient (C)', 'Hospital Patient (D)']
['1', 'ADRENALINE 1:1000/ML INJ', '1', '100', '1']
['2', 'ATRACURIUM 25MG INJ (2.3ML)', '1', '1', '1']
['3', 'ATRACURIUM 50MG INJ (5ML)', '1', '1', '1']
['4', 'BUVPIVACAINE 0.5%(HEAVY MERCAINE) INJ', '2', '2', '1']


In [3]:
# --- Cell 3: Parse categories + tables into one normalized table ---

def clean_category(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"^Listing\s+of\s+", "", s, flags=re.IGNORECASE).strip()
    return s.title() if s else "Uncategorized"

def normalize_header(cols):
    out = []
    for c in cols:
        c = (c or "").strip()
        c = re.sub(r"\s+", " ", c)
        out.append(c)
    # drop trailing empty header cells
    while out and out[-1] == "":
        out.pop()
    return out

def parse_money(cell: str):
    """Return numeric float if parseable, else None."""
    if cell is None:
        return None
    s = str(cell).strip().strip('"').strip("'")
    if s == "":
        return None

    # Remove currency and commas (accept ₦ or N or NGN etc)
    s2 = (s.replace("₦", "")
            .replace("NGN", "")
            .replace("N", "")
            .replace(",", "")
            .strip())

    # Some cells might contain text; accept only number-like
    if not re.fullmatch(r"-?\d+(?:\.\d+)?", s2):
        return None

    try:
        return float(s2)
    except:
        return None

def format_naira(v: float) -> str:
    return f"₦{v:,.2f}"

current_category = "Uncategorized"
current_header = None
records = []

for r in rows:
    # Skip completely empty rows
    if not any(str(c).strip() for c in r):
        continue

    first = str(r[0]).strip()

    # Category marker
    if re.search(r"\bListing\s+of\b", first, flags=re.IGNORECASE):
        current_category = clean_category(first)
        current_header = None
        continue

    # Header row marker (table start)
    if first.upper() == "S/N" and len(r) > 1 and str(r[1]).strip().lower() == "name":
        current_header = normalize_header(r)
        continue

    # Data row (requires a header and a numeric S/N)
    if current_header:
        # Accept integers stored as "1" or "1.0"
        sn_raw = first
        if re.fullmatch(r"\d+(?:\.0+)?", sn_raw):
            sn = int(float(sn_raw))
        else:
            continue

        name = str(r[1]).strip() if len(r) > 1 else ""
        if not name:
            continue

        price_cols = current_header[2:]  # dynamic columns after S/N + Name
        prices = {}

        for idx, col_name in enumerate(price_cols, start=2):
            cell = r[idx] if idx < len(r) else ""
            num = parse_money(cell)

            # Decide N/A rules
            if num is None:
                val = "N/A" if BLANK_TO_NA else ""
            else:
                if (ZERO_TO_NA and abs(num) < 1e-12) or (BELOW_100_TO_NA and num < 100):
                    val = "N/A"
                else:
                    val = format_naira(num) if FORMAT_NAIRA else str(num)

            prices[col_name] = val

        rec = {"S/N": sn, "Name": name, "Category": current_category, **prices}
        records.append(rec)

df = pd.DataFrame(records)

print("Parsed rows:", len(df))
print("Categories:", df["Category"].nunique() if len(df) else 0)
df.head(10)


Parsed rows: 1970
Categories: 35


,S/N,Name,Category,Outsourced (B),Walk in Patient (C),Hospital Patient (D)
0,1,ADRENALINE 1:1000/ML INJ,Anaesthetics/Anxiolytics/Emergency Drugs,N/A,₦100.00,N/A
1,2,ATRACURIUM 25MG INJ (2.3ML),Anaesthetics/Anxiolytics/Emergency Drugs,N/A,N/A,N/A
2,3,ATRACURIUM 50MG INJ (5ML),Anaesthetics/Anxiolytics/Emergency Drugs,N/A,N/A,N/A
3,4,BUVPIVACAINE 0.5%(HEAVY MERCAINE) INJ,Anaesthetics/Anxiolytics/Emergency Drugs,N/A,N/A,N/A
4,5,BUVPIVACAINE 0.5%(LIGHT MERCAINE) INJ,Anaesthetics/Anxiolytics/Emergency Drugs,N/A,N/A,N/A
5,6,DEXAMETHASONE 4MG/ML INJ,Anaesthetics/Anxiolytics/Emergency Drugs,N/A,N/A,N/A
6,7,DIAZEPAM INJ 10MG/2ML(VALIUM),Anaesthetics/Anxiolytics/Emergency Drugs,N/A,N/A,N/A
7,8,DOPAMINE 200MG/5ML INJ,Anaesthetics/Anxiolytics/Emergency Drugs,N/A,N/A,N/A
8,9,EPHEDRINE 3MG/ML INJ,Anaesthetics/Anxiolytics/Emergency Drugs,N/A,N/A,N/A
9,10,HALOTHANE INJ 250ML SOLUTION,Anaesthetics/Anxiolytics/Emergency Drugs,N/A,N/A,N/A


In [4]:
# --- Cell 4: Sort + write cleaned CSV (safe overwrite, Excel-safe ₦) ---

from pathlib import Path
import tempfile
import os

if df.empty:
    raise ValueError("No records parsed. Check the input file format in drugs.csv/drugs.xlsx.")

# Column order: S/N, Name, Category, then all other columns (dynamic)
base_cols = ["S/N", "Name", "Category"]
other_cols = [c for c in df.columns if c not in base_cols]
df = df[base_cols + other_cols]

# Sort like your other cleaned files
df = df.sort_values(["Category", "S/N"], kind="stable").reset_index(drop=True)

summary = df.groupby("Category")["Name"].count().sort_values(ascending=False)
print("Rows per category:")
print(summary)

# ✅ Write UTF-8 with BOM (utf-8-sig) so Excel shows ₦ correctly
tmp_path = Path(tempfile.mkstemp(suffix=".csv")[1])
df.to_csv(tmp_path, index=False, lineterminator="\n", encoding="utf-8-sig")

try:
    os.replace(tmp_path, OUTPUT_CSV)
    print(f"Saved cleaned CSV to: {OUTPUT_CSV.resolve()}")
except PermissionError:
    fallback = OUTPUT_CSV.with_name(f"{OUTPUT_CSV.stem}_CLOSE_EXCEL_AND_RERUN{OUTPUT_CSV.suffix}")
    os.replace(tmp_path, fallback)
    print("\n⚠️ Could not overwrite output file (likely open in Excel).")
    print("Saved instead to:", fallback.resolve())

df.head(20)


Rows per category:
Category
Anaesthetics/Anxiolytics/Emergency Drugs                                                                 647
Antibacterial /Antibiotics                                                                               190
Anti-Malaria                                                                                             130
Anti-Hypertensive & Cardiovascular Drugs                                                                 124
Analgesics, Antipyretics And Non-Steroidal Anti-Inflamatory Drugs                                        105
Haematinics, Anti-Coagulants,Antifibrinolytics,Vitamins, Minerals And Supplements                         80
Opthalmological Drugs                                                                                     68
Antacids & Anti-Ulcer Drugs                                                                               64
Hormones,Contaceptives, Genito-Urinary Disorders,Obstetrics & Oxytocics Drugs                       

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\ACEMX\\AppData\\Local\\Temp\\tmpiz01uic6.csv' -> 'drugs_new_CLOSE_EXCEL_AND_RERUN.csv'